## Model Training On Real Data

In [ ]:
# import modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

### Load Dataset

In [ ]:
transactions_data = pd.read_csv("data/credit_card_data_real.zip", compression="zip")
print(transactions_data.head())

## Class Imbalance

As it can be seen here there is a class imbalance poblem with the dataset. The fraudulent transactions are only 492 or less than 1% of the entire dataset

In [ ]:
print(transactions_data["Class"].value_counts())
print()
print(transactions_data["Class"].value_counts(normalize=True))

In [ ]:
plt.bar(transactions_data["Class"].value_counts().index, transactions_data["Class"].value_counts().values, color=["blue", "orange"])
plt.xticks([0, 1], ["Legit", "Fraud"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Target Labels Distribution")
plt.show()

## Data Preparation

### Data Resampling

As seen before the data has to be resampled to mitigate the class imbalance problem. SMOTE is used again to over-sample the minority class

In [ ]:
from imblearn.over_sampling import SMOTE

X = transactions_data.drop(["Class"], axis=1)
y = transactions_data["Class"]

smote = SMOTE(random_state=42)

X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
plt.bar(y_resampled.value_counts().index, y_resampled.value_counts().values, color=["blue", "orange"])
plt.xticks([0, 1], ["Legit", "Fraud"])
plt.xlabel("Class")
plt.ylabel("Count")
plt.title("Target Labels Distribution")
plt.show()

## Model Building

It's time to try a few machine learning models and see how they perform on real data.

In [ ]:
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

### Evaluation Metrics

Given the imbalnced dataset, some suitable metrics are __ROC AUC__ score, __average precision__ score and __f1__ score.

- **ROC AUC**: is the area under the Receiver Operating Characteristic curve

- **average precision**: summarizes a precision-recall curve as the weighted mean of precisions achieved at each threshold, with the increase in recall from the previous threshold used as the weight:

$$ AP = \sum(R_n - R_{n - 1})P_n $$

  where $P_n$ and $ R_n $ are the precision and recall at the nth threshold 

- **f1 score**: is the harmonic mean of precision and recall
$$ F1 = \frac{2 * TP}{2 * TP + FP + FN} $$

### Logistic Regression

Bellow a pipeline was created with StandardScaler, SMOTE over-sampling and a LogisticRegression model. The model was trained using stratified cross-validation and achived a roc_aus score of 98%, average precision score of 74% and f1 score of 11%

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate

# Define components 
scaler = StandardScaler()
log_reg = LogisticRegression(random_state=42, class_weight="balanced")

# Define pipeline
log_reg_pipeline = ImbPipeline(
    steps=[
        ("scaler", scaler),
        ("smote", smote),
        ("model", log_reg)
    ]
)

# Cross-validation Iterator
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Metrics
scoring = {
    "roc_aus": "roc_auc",
    "pr_auc": "average_precision",
    "f1": "f1"
}

# Run cv
results = cross_validate(
    log_reg_pipeline,
    X, y,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    return_estimator=True,
    return_indices=True
)

for metric in scoring:
    print(f"{metric}: {results["test_" + metric].mean():.4f}")

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_cv_results(results, X, y)
plt.show()

### Stochastic Gradient Descent Classifier

Bellow a pipeline was created with StandardScaler, SMOTE over-sampling and a SGDClassifier model. The model was trained using stratified cross-validation and achived a roc_aus score of 98%, average precision score of 73% and f1 score of 10%

In [ ]:
sgd_clf = SGDClassifier(random_state=42, class_weight="balanced")

sgd_pipeline = ImbPipeline(
    steps=[
        ("scaler", scaler),
        ("smote", smote),
        ("model", sgd_clf)
    ]
)

results = cross_validate (
    sgd_pipeline,
    X, y, 
    scoring=scoring,
    cv=cv,
    return_train_score=False,
    return_estimator=True,
    return_indices=True
)

for metric in scoring:
    print(f"{metric}: {results["test_" + metric].mean():.4f}")

In [ ]:
from sklearn.metrics import roc_curve, auc, RocCurveDisplay

RocCurveDisplay.from_cv_results(results, X, y)
plt.show()

### Gradient Boosting Classifier

The Gradient Boosting Classifier pipieline achived a roc_auc score of 97%, 83% average precision score and 66% f1 score. This is the best model so far although it takes longer to train

In [ ]:
gb_clf = HistGradientBoostingClassifier(random_state=42)

gb_clf_pipeline = ImbPipeline(
    steps=[
        ("scaler", scaler),
        ("smote", smote),
        ("model", gb_clf)
    ]
)

results = cross_validate(
    gb_clf_pipeline,
    X, y,
    scoring=scoring,
    cv=cv,
    return_train_score=False,
    return_estimator=True,
    return_indices=True
)

for metric in scoring:
    print(f"{metric}: {results["test_" + metric].mean():.4f}")